# Check image-to-label pairs

This notebook checks whether every `.jpg` image in `train`, `val`, and `test` has a same-named YOLO `.txt` label. It also finds label files that do not have an image.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works when the notebook is launched from either the repository root
# or the notebooks directory. Change this if your dataset is elsewhere.
DATASET_ROOT = Path('detection_dataset')
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path('../detection_dataset')
DATASET_ROOT = DATASET_ROOT.resolve()

SPLITS = ('train', 'val', 'test')
print(f'Dataset: {DATASET_ROOT}')

Dataset: /home/mehedinaeem/Desktop/Code/Bitol_Computer_Vision_System/detection_dataset


In [ ]:
def check_split(dataset_root: Path, split: str):
    image_dir = dataset_root / 'images' / split
    label_dir = dataset_root / 'labels' / split

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f'Missing directory for {split}: {image_dir} or {label_dir}'
        )

    images = {p.stem: p for p in image_dir.glob('*.jpg')}
    labels = {p.stem: p for p in label_dir.glob('*.txt')}
    all_stems = sorted(images.keys() | labels.keys())

    rows = []
    for stem in all_stems:
        image = images.get(stem)
        label = labels.get(stem)
        if image and label:
            status = 'matched'
        elif image:
            status = 'missing_label'
        else:
            status = 'missing_image'

        rows.append({
            'split': split,
            'stem': stem,
            'image_file': image.name if image else None,
            'label_file': label.name if label else None,
            'status': status,
        })

    return pd.DataFrame(rows)

pair_report = pd.concat(
    [check_split(DATASET_ROOT, split) for split in SPLITS],
    ignore_index=True,
)
pair_report.head()

,split,stem,image_file,label_file,status
0,train,classes,NaN,classes.txt,missing_image
1,train,healthy_0001,healthy_0001.jpg,healthy_0001.txt,matched
2,train,healthy_0002,healthy_0002.jpg,healthy_0002.txt,matched
3,train,healthy_0003,healthy_0003.jpg,healthy_0003.txt,matched
4,train,healthy_0004,healthy_0004.jpg,healthy_0004.txt,matched


In [ ]:
summary = (
    pair_report.groupby('split', sort=False)
    .agg(
        jpg_images=('image_file', 'count'),
        txt_labels=('label_file', 'count'),
        matched_pairs=('status', lambda s: (s == 'matched').sum()),
        missing_labels=('status', lambda s: (s == 'missing_label').sum()),
        orphan_labels=('status', lambda s: (s == 'missing_image').sum()),
    )
    .reindex(SPLITS)
)
summary['all_images_have_labels'] = summary['missing_labels'].eq(0)
summary['all_labels_have_images'] = summary['orphan_labels'].eq(0)
display(summary)

,jpg_images,txt_labels,matched_pairs,missing_labels,orphan_labels,all_images_have_labels,all_labels_have_images
split,,,,,,,
train,1698,1298,1297,401,1,False,False
val,486,487,486,0,1,True,False
test,248,249,248,0,1,True,False


In [ ]:
problems = pair_report[pair_report['status'] != 'matched'].reset_index(drop=True)

if problems.empty:
    print('OK: every .jpg image has a matching .txt label, and every label has an image.')
else:
    print(f'Found {len(problems)} unmatched file(s):')
    display(problems)

Found 404 unmatched file(s):


,split,stem,image_file,label_file,status
0,train,classes,NaN,classes.txt,missing_image
1,train,healthy_0453,healthy_0453.jpg,NaN,missing_label
2,train,healthy_0842,healthy_0842.jpg,NaN,missing_label
3,train,healthy_1049,healthy_1049.jpg,NaN,missing_label
4,train,unhealthy_0452,unhealthy_0452.jpg,NaN,missing_label
...,...,...,...,...,...
399,train,unhealthy_1018,unhealthy_1018.jpg,NaN,missing_label
400,train,unhealthy_1020,unhealthy_1020.jpg,NaN,missing_label
401,train,unhealthy_1021,unhealthy_1021.jpg,NaN,missing_label
402,val,classes,NaN,classes.txt,missing_image


In [ ]:
# Optional: save every file's matching status for later inspection.
output_path = Path('image_label_pair_report.csv')
pair_report.to_csv(output_path, index=False)
print(f'Saved report to: {output_path.resolve()}')